### ENSIMAG – Grenoble INP – UGA - Academic year 2025-2026
# Introduction to Statistical Learning and Applications ([website](https://github.com/ISLA-Grenoble/2025-main))

- Pedro L. C. Rodrigues -- `pedro.rodrigues@inria.fr`

- Isabella Costa Maia -- `isabella.costa-maia@grenoble-inp.fr`

- Pierre Marrec -- `pierre.marrec@inria.fr`

***

### ⚠️ General guidelines for TPs

Each team shall upload its report on their repository before the deadline indicated at the course website. Please
**include the name of all members** of the team on top of your report.
The report should contain graphical representations and explanatory text. For each graph, axis names should be provided as well
as a legend when it is appropriate. Figures should be explained by a few sentences in the text. Answer to
the questions in order and refer to the question number in your report. Computations and
graphics have to be performed in `python`. The report should be written as a jupyter notebook. This is a file format that allows users to format documents containing text written in markdown and `python` instructions. You should include all of the `python` instructions that you have used in the document so that it may be possible to replicate your results.

***

# 🖥️ TP3: Benchmarking classification methods

In this TP, we will be using mostly the packages `numpy`, `sklearn`, and `matplotlib`.

## ▶️ Part 1: Simulated data

Consider a simulated dataset generated as follows:

----
### -- Step 1
For each data point $i$, sample its label from a Bernoulli distribution $y_i \sim \mathcal{B}(p)$, i.e. $y_i = 1$ with probability $p$ and $y_i = 0$ with probability $1-p$. Note that to sample a random variable $B$ from $\mathcal{B}(p)$ you can first sample $U$ from an uniform distribution as in `U = numpy.random.rand()` and then note that $B = \mathbf{1}(U < p)$ where $\mathbf{1}(\cdot)$ is an indicator function.

### -- Step 2

Then, depending on the label $y_i \in \{0, 1\}$ the associated data point $\mathbf{x}_i \in \mathbb{R}^2$ is sampled as follows:

$$
  \mathbf{x}_i \mid y_i = 0 \sim \mathcal{N}(\boldsymbol{\mu}_0, \boldsymbol{\Sigma}_0) \quad \text{and} \quad \mathbf{x}_i \mid y_i = 1 \sim \mathcal{N}(\boldsymbol{\mu}_1, \boldsymbol{\Sigma}_1)
$$

where $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ is a multivariate normal distribution with mean $\boldsymbol{\mu}$ and covariance matrix $\boldsymbol{\Sigma}$ with pdf

$$
p_{\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})}(x) = \dfrac{1}{2\pi\sqrt{\det{\boldsymbol{\Sigma}}}}\exp\left(-\dfrac{1}{2}\big(\boldsymbol{x}-\boldsymbol{\mu}\big)^\top \boldsymbol{\Sigma}^{-1}\big(\boldsymbol{x}-\boldsymbol{\mu}\big)\right)
$$
and
$$
\boldsymbol{\mu}_0 = \left[\begin{array}{c}0 \\ 0\end{array}\right] \quad \boldsymbol{\mu}_1 = \left[\begin{array}{c}\varepsilon \\ 0\end{array}\right] \quad \boldsymbol{\Sigma}_0 = \left[\begin{array}{cc}0.5 & 0 \\ 0 & 0.5\end{array}\right] \quad \boldsymbol{\Sigma}_1 = \left[\begin{array}{cc}0.4 & 0 \\ 0 & 0.4\end{array}\right]
$$

Note that to sample a $p$-dimensional vector $\mathbf{x}$ from $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$, you can use function `numpy.random.multivariate_normal`.

----

We will denote a set of $N$ data points $\{(\mathbf{x}_i, y_i)\}_{i = 1}^N$ simulated with $\varepsilon$ and $p$ as $\mathcal{D}(N \mid \varepsilon, p)$. 

Define two datasets:
$$
\mathcal{D}_\text{train} = \mathcal{D}(50 \mid 2, 0.30) \quad \text{and} \quad \mathcal{D}_{\text{test}} = \mathcal{D}(10^3 \mid 2, 0.30)~.
$$

**(a)** Plot the data points in $\mathcal{D}_\text{train} \cup \mathcal{D}_\text{test}$ using different colors to indicate the classes of each data point and different pointing symbols to indicate whether a point is from the train or test set. You should use `matplotlib`'s function for scatterplots. Remember to always include a legend in your figure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng

GLOBAL_RNG = default_rng(42)


def generate_data(epsilon, p, train_len, test_len, rng=None):
    """Generate train/test data from the TP Gaussian model."""
    if rng is None:
        rng = default_rng()

    mu0 = np.array([0.0, 0.0])
    mu1 = np.array([epsilon, 0.0])
    cov0 = np.array([[0.5, 0.0], [0.0, 0.5]])
    cov1 = np.array([[0.4, 0.0], [0.0, 0.4]])

    def sample_split(n):
        y = rng.binomial(n=1, p=p, size=n).astype(int)
        X = np.empty((n, 2))

        mask1 = y == 1
        mask0 = ~mask1

        X[mask0] = rng.multivariate_normal(mu0, cov0, size=mask0.sum())
        X[mask1] = rng.multivariate_normal(mu1, cov1, size=mask1.sum())
        return X, y

    X_train, y_train = sample_split(train_len)
    X_test, y_test = sample_split(test_len)

    parameters = {
        "mu0": mu0,
        "mu1": mu1,
        "cov0": cov0,
        "cov1": cov1,
        "P0": 1 - p,
        "P1": p,
    }
    return X_train, y_train, X_test, y_test, parameters


X_train, y_train, X_test, y_test, _ = generate_data(
    epsilon=2, p=0.30, train_len=50, test_len=1000, rng=GLOBAL_RNG
)

plt.figure(figsize=(8, 6))
colors = {0: "tab:red", 1: "tab:blue"}

for c in [0, 1]:
    plt.scatter(
        X_test[y_test == c, 0],
        X_test[y_test == c, 1],
        marker="x",
        s=20,
        alpha=0.45,
        color=colors[c],
        label=f"test - class {c}",
    )
    plt.scatter(
        X_train[y_train == c, 0],
        X_train[y_train == c, 1],
        marker="o",
        s=60,
        edgecolor="black",
        linewidth=0.4,
        color=colors[c],
        label=f"train - class {c}",
    )

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Simulated data: train and test sets")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


**(b)** What is the mathematical expression for the optimal Bayes classifier in this setting? And for its boundary region? Remember that the Bayes classifier can be written in terms of the ratio of $\text{Prob}(Y = 1 \mid \mathbf{x})$ over $\text{Prob}(Y = 0 \mid \mathbf{x})$ and that the values of $\mathbf{x} \in \mathbb{R}^2$ for which this ratio is 1 are those defining its boundary. Beware, however, that in this exercise we're considering $\text{Prob}(Y = 1) = p$ and $\text{Prob}(Y = 0) = 1-p$, so they are not necessarily always equal.

**Response (b):**

The Bayes classifier predicts class 1 when the posterior odds are at least 1:
$$
g^*(x)=
\begin{cases}
1 & \text{if } \dfrac{\mathbb{P}(Y=1\mid X=x)}{\mathbb{P}(Y=0\mid X=x)} \ge 1, \\
0 & \text{otherwise.}
\end{cases}
$$

Using Bayes' rule with $\mathbb{P}(Y=1)=p$ and $\mathbb{P}(Y=0)=1-p$:
$$
\dfrac{\mathbb{P}(Y=1\mid X=x)}{\mathbb{P}(Y=0\mid X=x)}
=
\dfrac{p\, f_1(x)}{(1-p)\, f_0(x)},
$$
where $f_k(x)$ is the Gaussian density of class $k$.

Equivalent log form:
$$
g^*(x)=1
\iff
\log\!\left(\frac{p}{1-p}\right)
+
\log\!\left(\frac{f_1(x)}{f_0(x)}\right)\ge 0.
$$

The decision boundary region is therefore
$$
\mathcal{B}=\left\{x\in\mathbb{R}^2:\; p\,f_1(x)=(1-p)\,f_0(x)\right\}.
$$

Since $\Sigma_0 \neq \Sigma_1$, this boundary is quadratic (QDA-type).


**(c)** Implement a Bayes classifier for this setup using scikit-learn's API as explained [here](https://scikit-learn.org/stable/developers/develop.html). This means that you will be writing a new classifier that follows the same logic and API as scikit-learn, but does not exist in the package. Use your implementation to estimate the error of the Bayes classifier on the samples from $\mathcal{D}(10^4 \mid 2, 0.3)$. How would you expect your results to change for other values of $\varepsilon$? Plot a curve showing how the Bayes error rate changes for different choices $\varepsilon$ (note that you will have to generate new datasets for this).

**Response (c):**
We implement a Bayes classifier compatible with scikit-learn's API.
If the true parameters are provided (`mu`, `Sigma`, priors), the classifier behaves as an oracle Bayes rule.
Otherwise, `fit` estimates these parameters from data and `predict` uses the same discriminant scores.


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted


class BayesClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        mu0=None,
        mu1=None,
        Sigma0=None,
        Sigma1=None,
        pi0=0.5,
        pi1=0.5,
        classes=(0, 1),
    ):
        self.mu0 = mu0
        self.mu1 = mu1
        self.Sigma0 = Sigma0
        self.Sigma1 = Sigma1
        self.pi0 = pi0
        self.pi1 = pi1
        self.classes = classes

    @staticmethod
    def _log_gaussian_density(X, mu, Sigma):
        sign, logdet = np.linalg.slogdet(Sigma)
        if sign <= 0:
            raise ValueError("Covariance matrix must be positive definite.")

        inv = np.linalg.inv(Sigma)
        diff = X - mu
        quad = np.einsum("ij,jk,ik->i", diff, inv, diff)
        d = X.shape[1]
        return -0.5 * (quad + logdet + d * np.log(2 * np.pi))

    def _use_oracle(self):
        return all(
            v is not None for v in (self.mu0, self.mu1, self.Sigma0, self.Sigma1)
        )

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        classes = np.unique(y)
        if len(classes) != 2:
            raise ValueError("This implementation only supports binary classification.")

        self.classes_ = classes
        X0 = X[y == classes[0]]
        X1 = X[y == classes[1]]

        self.mu0_ = X0.mean(axis=0)
        self.mu1_ = X1.mean(axis=0)
        self.Sigma0_ = np.cov(X0, rowvar=False)
        self.Sigma1_ = np.cov(X1, rowvar=False)
        self.pi0_ = len(X0) / len(X)
        self.pi1_ = len(X1) / len(X)
        return self

    def _parameters(self):
        if self._use_oracle():
            classes = np.asarray(self.classes)
            if classes.shape[0] != 2:
                raise ValueError("`classes` must contain exactly two labels.")
            return (
                np.asarray(self.mu0),
                np.asarray(self.mu1),
                np.asarray(self.Sigma0),
                np.asarray(self.Sigma1),
                float(self.pi0),
                float(self.pi1),
                classes,
            )

        check_is_fitted(
            self,
            ["mu0_", "mu1_", "Sigma0_", "Sigma1_", "pi0_", "pi1_", "classes_"],
        )
        return (
            self.mu0_,
            self.mu1_,
            self.Sigma0_,
            self.Sigma1_,
            self.pi0_,
            self.pi1_,
            self.classes_,
        )

    def decision_function(self, X):
        X = check_array(X)
        mu0, mu1, Sigma0, Sigma1, pi0, pi1, _ = self._parameters()

        if pi0 <= 0 or pi1 <= 0:
            raise ValueError("Class priors must be strictly positive.")

        s0 = self._log_gaussian_density(X, mu0, Sigma0) + np.log(pi0)
        s1 = self._log_gaussian_density(X, mu1, Sigma1) + np.log(pi1)
        return s1 - s0

    def predict(self, X):
        _, _, _, _, _, _, classes = self._parameters()
        scores = self.decision_function(X)
        return np.where(scores >= 0, classes[1], classes[0])


def calculate_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(y_true != y_pred)


In [ ]:
_, _, X_test_big, y_test_big, params = generate_data(
    epsilon=2, p=0.3, train_len=0, test_len=10_000, rng=default_rng(1)
)

bayes_model = BayesClassifier(
    mu0=params["mu0"],
    mu1=params["mu1"],
    Sigma0=params["cov0"],
    Sigma1=params["cov1"],
    pi0=params["P0"],
    pi1=params["P1"],
)

predictions = bayes_model.predict(X_test_big)
print(f"Estimated Bayes error on D(10^4 | 2, 0.3): {calculate_error(y_test_big, predictions):.4f}")


As $\varepsilon$ increases, the class means move further apart, so the overlap between the two Gaussian clouds decreases.
Therefore, the Bayes error (irreducible classification error) is expected to decrease with $\varepsilon$.


In [ ]:
epsilon_grid = np.linspace(0, 6, 25)
bayes_errors = []

for eps in epsilon_grid:
    mc_errors = []
    for seed in range(12):
        _, _, X_test_eps, y_test_eps, params_eps = generate_data(
            epsilon=eps, p=0.3, train_len=0, test_len=4000, rng=default_rng(1000 + seed)
        )
        bayes_eps = BayesClassifier(
            mu0=params_eps["mu0"],
            mu1=params_eps["mu1"],
            Sigma0=params_eps["cov0"],
            Sigma1=params_eps["cov1"],
            pi0=params_eps["P0"],
            pi1=params_eps["P1"],
        )
        mc_errors.append(calculate_error(y_test_eps, bayes_eps.predict(X_test_eps)))
    bayes_errors.append(np.mean(mc_errors))

plt.figure(figsize=(8, 5))
plt.plot(epsilon_grid, bayes_errors, marker="o")
plt.xlabel(r"$\varepsilon$")
plt.ylabel("Estimated Bayes error")
plt.title("Bayes error as a function of class separation")
plt.grid(alpha=0.25)
plt.show()


**(d)** Given the structure of the model generating the datasets, which classifier presented in our lectures seems to be the most adequate? Justify your answer in terms of the assumptions behind the construction of each classifier.

**Response (d):**
The most adequate classifier is **QDA**.
Reason: each class is Gaussian, but the covariance matrices are different ($\Sigma_0 
eq \Sigma_1$).
QDA matches exactly this generative assumption and can represent a quadratic decision boundary, unlike LDA (shared covariance) or logistic regression (discriminative linear log-odds).


**(e)** Using `sklearn`, train a LDA, a QDA, and a logistic regression classifier on $\mathcal{D}_\text{train}$ and estimate their errors on the samples from $\mathcal{D}_\text{test}$. How do their errors compare to the value obtained in (c)? Can we expect the gap between the Bayes error rate and test error for each classifier change when the number of samples in $\mathcal{D}_{\text{train}}$ in change? Justify your answer both theoretically and empirically.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis

X_train, y_train, X_test, y_test, params = generate_data(
    epsilon=2, p=0.3, train_len=50, test_len=1000, rng=default_rng(2026)
)

bayes_model = BayesClassifier(
    mu0=params["mu0"],
    mu1=params["mu1"],
    Sigma0=params["cov0"],
    Sigma1=params["cov1"],
    pi0=params["P0"],
    pi1=params["P1"],
)

lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis()
lr = LogisticRegression(max_iter=3000, random_state=0)

models = {
    "Bayes (oracle)": bayes_model,
    "LDA": lda,
    "QDA": qda,
    "Logistic Regression": lr,
}

errors = {}

for name, model in models.items():
    if name != "Bayes (oracle)":
        model.fit(X_train, y_train)
    preds = model.predict(X_test)
    errors[name] = calculate_error(y_test, preds)

errors


**Response (e):**

- The Bayes classifier gives the smallest achievable error because it uses the true data-generating parameters.
- QDA should be the closest competitor, since it matches the correct family (Gaussian classes with class-specific covariance).
- LDA and logistic regression are misspecified here:
  - LDA enforces equal covariances.
  - Logistic regression imposes a linear log-odds boundary.

To make the comparison more stable (and avoid one-shot randomness), we run repeated simulations for several training sizes, then report:

1. the mean test error for each method;
2. the mean excess error with respect to Bayes.


In [ ]:
n_train_values = np.array([30, 60, 120, 250, 500, 1000, 2500])
n_test = 4000
n_repeats = 60

model_names = ["Bayes (oracle)", "LDA", "QDA", "Logistic Regression"]
mean_errors = {name: [] for name in model_names}
std_errors = {name: [] for name in model_names}
mean_gaps = {name: [] for name in ["LDA", "QDA", "Logistic Regression"]}

for n_train in n_train_values:
    trial_errors = {name: [] for name in model_names}

    for rep in range(n_repeats):
        # Retry generation until each class has enough samples for stable covariance estimation.
        for attempt in range(200):
            local_rng = default_rng(500_000 + 10_007 * rep + 37 * int(n_train) + attempt)
            X_train_n, y_train_n, X_test_n, y_test_n, params_n = generate_data(
                epsilon=2, p=0.3, train_len=int(n_train), test_len=n_test, rng=local_rng
            )
            class_counts = np.bincount(y_train_n, minlength=2)
            if class_counts.min() >= 3:
                break

        bayes_n = BayesClassifier(
            mu0=params_n["mu0"],
            mu1=params_n["mu1"],
            Sigma0=params_n["cov0"],
            Sigma1=params_n["cov1"],
            pi0=params_n["P0"],
            pi1=params_n["P1"],
        )
        bayes_error = calculate_error(y_test_n, bayes_n.predict(X_test_n))

        lda_n = LinearDiscriminantAnalysis().fit(X_train_n, y_train_n)
        qda_n = QuadraticDiscriminantAnalysis().fit(X_train_n, y_train_n)
        lr_n = LogisticRegression(max_iter=3000, random_state=0).fit(X_train_n, y_train_n)

        trial_errors["Bayes (oracle)"].append(bayes_error)
        trial_errors["LDA"].append(calculate_error(y_test_n, lda_n.predict(X_test_n)))
        trial_errors["QDA"].append(calculate_error(y_test_n, qda_n.predict(X_test_n)))
        trial_errors["Logistic Regression"].append(
            calculate_error(y_test_n, lr_n.predict(X_test_n))
        )

    for name in model_names:
        values = np.asarray(trial_errors[name])
        mean_errors[name].append(values.mean())
        std_errors[name].append(values.std())

    bayes_values = np.asarray(trial_errors["Bayes (oracle)"])
    for name in ["LDA", "QDA", "Logistic Regression"]:
        mean_gaps[name].append((np.asarray(trial_errors[name]) - bayes_values).mean())


# Plot 1: mean test error curves (easy to compare with TP33-style plots)
plt.figure(figsize=(8, 5))
for name in model_names:
    plt.plot(n_train_values, mean_errors[name], marker="o", label=name)

plt.xscale("log")
plt.xlabel(r"$N_{train}$ (log scale)")
plt.ylabel("Mean test error")
plt.title("Mean test error vs training size")
plt.grid(True, which="both", alpha=0.25)
plt.legend()
plt.show()


# Plot 2: mean excess error over Bayes
plt.figure(figsize=(8, 5))
for name, values in mean_gaps.items():
    plt.plot(n_train_values, values, marker="o", label=name)

plt.xscale("log")
plt.xlabel(r"$N_{train}$ (log scale)")
plt.ylabel("Mean excess error over Bayes")
plt.title("Gap to Bayes vs training size")
plt.grid(True, which="both", alpha=0.25)
plt.legend()
plt.show()


The two plots confirm the expected behavior:

- On the **mean test error** plot, all methods improve with larger training sets.
- On the **gap-to-Bayes** plot, QDA tends to approach Bayes fastest, while LDA/logistic regression keep a small residual gap due to model bias.

Small non-monotonic fluctuations are normal because results are estimated empirically from finite samples.


**(f)** Consider a new test set defined as $\mathcal{D}'_\text{test} = \mathcal{D}(1000 \mid 0.5, 0.7)$. Use the same classifiers trained in (e) and estimate their new test errors. Do you observe any difference in the results? Can you explain what is happening?

In [ ]:
# New test distribution D'(1000 | 0.5, 0.7)
_, _, X_test_shift, y_test_shift, params_shift = generate_data(
    epsilon=0.5, p=0.7, train_len=0, test_len=1000, rng=default_rng(999)
)

bayes_shift = BayesClassifier(
    mu0=params_shift["mu0"],
    mu1=params_shift["mu1"],
    Sigma0=params_shift["cov0"],
    Sigma1=params_shift["cov1"],
    pi0=params_shift["P0"],
    pi1=params_shift["P1"],
)

errors_shift = {
    "LDA": calculate_error(y_test_shift, lda.predict(X_test_shift)),
    "QDA": calculate_error(y_test_shift, qda.predict(X_test_shift)),
    "Logistic Regression": calculate_error(y_test_shift, lr.predict(X_test_shift)),
    "Bayes (oracle on shifted distribution)": calculate_error(
        y_test_shift, bayes_shift.predict(X_test_shift)
    ),
}

errors_shift


**Response (f):**

Yes, the errors increase on $\mathcal{D}'_{\text{test}}$.
Main reasons:

1. **Distribution shift:** models were trained on $(\varepsilon=2,\; p=0.3)$ and evaluated on $(\varepsilon=0.5,\; p=0.7)$.
2. **Lower separation:** $\varepsilon=0.5$ increases class overlap, so even the Bayes error is higher.
3. **Prior change:** the class balance changed from 30/70 to 70/30, which affects the optimal threshold and posterior odds.

So the degradation is expected and illustrates sensitivity to dataset shift.


## ▶️ Part 2: Real data

In this part we will consider the Titanic dataset available [here](https://www.kaggle.com/competitions/titanic/data). The goal here will be to build a machine learning model that predicts which passengers survived the Titanic shipwreck. Each passenger (i.e., data point) is composed of a set of categorical and continuous features, and its labels are either 0 (dead) or 1 (survived).

First of all, you should download both the `training` and the `test` datasets.

-- The `training` set should be used to build your machine learning models. The labels for each passenger are provided. Your model will be based on “features” like passengers’ gender and class. You can also use feature engineering to create new features.

-- The `test` set should be used to see how well your model performs on unseen data. For the test set, we do not provide the ground truth for each passenger. It is your job to predict these outcomes. For each passenger in the test set, use the model you trained to predict whether or not they survived the sinking of the Titanic.

Follow the guidelines from [here](https://www.kaggle.com/competitions/titanic/overview) to understand how to submit the results of your predictions on the `test` set and obtain the score of your model.

### Suggestions:

-- Don't hesitate to do some exploratory data analysis before building your machine learning model. You chould check, for instance, which kind of cross-validator seems the most appropriate for assessing the score of your classifier : are the data points completely IID? are they ordered somehow? split into groups? Beware of all this.

-- Since you will be handling predictors with different data types, it might be useful to take a look at the concept of `ColumnTransformer` from scikit-learn [here](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html). You could also check these two videos about how to build complext pipelines [1](https://www.youtube.com/watch?v=7TZ7j4HSzmE) and [2](https://www.youtube.com/watch?v=lhMqqauXtW0).

-- Take a look at the package [`skrub`](https://skrub-data.org/stable/). You would be surprised with how easy it is to get a very good score on this dataset using `tabular_learner`.

**(a)** Explain the feature engineering that you had to do with the dataset. If you've used `skrub`, explain how the encoding for each kind of predictor was decided.

**Response (a): Feature engineering and preprocessing**

I used a tabular pipeline with explicit feature engineering before model training:

- `FamilySize = SibSp + Parch + 1`
- `IsAlone = 1(FamilySize = 1)`
- `Title` extracted from `Name` (Mr, Mrs, Miss, etc.)
- `Deck` extracted from the first letter of `Cabin` (`Unknown` when missing)
- `TicketPrefix` extracted from `Ticket` (non-numeric prefix)

Then I dropped raw high-cardinality text columns (`Name`, `Ticket`, `Cabin`) and kept engineered summaries.

Preprocessing choices:

- Numerical variables: median imputation + standardization.
- Categorical variables: most-frequent imputation + one-hot encoding with infrequent-category grouping (`handle_unknown='infrequent_if_exist'`).

This setup is robust to missing values and unseen categories in the Kaggle test set.


**(b)** What type of classifier did you end up using? Why? What was your score in the public leaderboard from Kaggle?

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier


def add_titanic_features(df):
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).fillna("Unknown")
    df["Deck"] = df["Cabin"].fillna("Unknown").str[0]
    ticket_prefix = (
        df["Ticket"]
        .fillna("")
        .str.replace(r"[./]", "", regex=True)
        .str.replace(r"\d+", "", regex=True)
        .str.strip()
        .replace("", "NONE")
    )
    df["TicketPrefix"] = ticket_prefix
    return df


candidate_dirs = [Path("."), Path("data"), Path("dataset"), Path("TP3"), Path("TP3/data")]
data_dir = next(
    (d for d in candidate_dirs if (d / "train.csv").exists() and (d / "test.csv").exists()),
    None,
)

if data_dir is None:
    print("train.csv and test.csv not found. Place them next to the notebook or in ./data, then rerun this cell.")
else:
    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")

    train_fe = add_titanic_features(train_df)
    test_fe = add_titanic_features(test_df)

    y = train_fe["Survived"].astype(int)

    drop_cols = ["Survived", "Name", "Ticket", "Cabin"]
    X = train_fe.drop(columns=drop_cols)
    X_kaggle = test_fe.drop(columns=["Name", "Ticket", "Cabin"])

    numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features and c != "PassengerId"]

    # Keep PassengerId only for submission.
    numeric_features = [c for c in numeric_features if c != "PassengerId"]

    preprocess = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="infrequent_if_exist",
                                min_frequency=0.01,
                            ),
                        ),
                    ]
                ),
                categorical_features,
            ),
        ],
        remainder="drop",
    )

    candidate_models = {
        "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
        "Random Forest": RandomForestClassifier(
            n_estimators=700,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=700,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_summary = {}

    for model_name, estimator in candidate_models.items():
        pipe = Pipeline([("prep", preprocess), ("model", estimator)])
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
        cv_summary[model_name] = {
            "cv_mean_accuracy": scores.mean(),
            "cv_std": scores.std(),
        }

    cv_table = pd.DataFrame(cv_summary).T.sort_values("cv_mean_accuracy", ascending=False)
    print(cv_table.round(4))

    best_name = cv_table.index[0]
    best_estimator = clone(candidate_models[best_name])

    final_model = Pipeline([("prep", preprocess), ("model", best_estimator)])
    final_model.fit(X, y)

    kaggle_pred = final_model.predict(X_kaggle).astype(int)

    submission = pd.DataFrame(
        {
            "PassengerId": test_fe["PassengerId"].astype(int),
            "Survived": kaggle_pred,
        }
    )
    submission_path = data_dir / "submission_tp3.csv"
    submission.to_csv(submission_path, index=False)

    print(f"Best model by 5-fold CV: {best_name}")
    print(
        f"Mean CV accuracy: {cv_table.iloc[0]['cv_mean_accuracy']:.4f} +/- "
        f"{cv_table.iloc[0]['cv_std']:.4f}"
    )
    print(f"Submission file written to: {submission_path}")


**Response (b):**
I selected the classifier with the best 5-fold stratified CV accuracy among the tested pipelines (logistic regression, random forest, extra trees), after the feature engineering described above.

The previous cell prints:
- the chosen model;
- its mean CV accuracy and standard deviation;
- the location of the generated Kaggle submission file (`submission_tp3.csv`).

After uploading the file to Kaggle, add your exact public leaderboard score here.
